# Visualize crops, positive pairs, and augmentations

Used to tune `crop_size` and per-augmentation strengths before any training. Each cell is independent — re-run individual cells after editing `cfg`.


In [ ]:
# Cell 1: pick a zarr, build the dataset, print key facts, define visualization helpers.
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np

from config import Config
from data.picker import pick_zarr_path
from data.dataset import ZarrCellDataset

cfg = Config()
if cfg.zarr_path is None:
    cfg.zarr_path = pick_zarr_path()

ds = ZarrCellDataset(cfg)

print('zarr_path           :', cfg.zarr_path)
print('arr.shape (T,C,Z,Y,X):', tuple(ds.arr.shape))
print('arr.dtype           :', ds.arr.dtype)
print('arr.chunks          :', ds.arr.chunks)
print('z_thickness         :', cfg.z_thickness)
print('Δt frames           :', (ds.delta_t_min, ds.delta_t_max))
print('z range slices      :', (ds.z_min, ds.z_max))


def mip(volume):
    """max intensity projection across z: (z, h, w) -> (h, w)."""
    return volume.max(axis=0)


def z_grid(volume, ncols=5):
    """tile a (z, h, w) volume into one 2d image, ncols slices per row."""
    z, h, w = volume.shape
    nrows = (z + ncols - 1) // ncols
    out = np.zeros((nrows * h, ncols * w), dtype=volume.dtype)
    for i in range(z):
        r, c = divmod(i, ncols)
        out[r*h:(r+1)*h, c*w:(c+1)*w] = volume[i]
    return out


In [ ]:
# Cell 2: pull one positive pair, show MIPs of view1 / view2 / diff in a 1x3 row.
import matplotlib.pyplot as plt

pair = next(iter(ds))
v1 = pair['view1'][0].numpy()   # (z, h, w)
v2 = pair['view2'][0].numpy()
meta = pair['meta']
print('meta:', meta)
print('view shape (z, h, w):', v1.shape)

# collapse each volume to a 2d mip so the panels stay compact; cell 2b shows full z structure.
v1_mip = mip(v1)
v2_mip = mip(v2)
diff_mip = v1_mip - v2_mip

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
panels = [
    (v1_mip,   f"view1 MIP, t={int(meta['t1'])}"),
    (v2_mip,   f"view2 MIP, t={int(meta['t2'])}"),
    (diff_mip, 'diff of MIPs (view1 - view2)'),
]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 2b: full z-slice grid for the RAW (un-augmented) crop, so we can inspect z structure.
import matplotlib.pyplot as plt

m = meta
crop_h = crop_w = cfg.crop_size
raw = ds._load_crop(int(m['t1']), int(m['z']), int(m['y']), int(m['x']), crop_h, crop_w)
raw_volume = raw[0].numpy()  # (z, h, w)

z0 = int(m['z'])
z1 = z0 + cfg.z_thickness

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(z_grid(raw_volume), cmap='gray')
ax.set_title(f'raw crop, all {cfg.z_thickness} z slices at t={int(m["t1"])}, z=[{z0}, {z1})')
ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 3: 4x4 grid of independently augmented variants of the SAME raw crop, each shown as a z-MIP.
import matplotlib.pyplot as plt
from data.augmentations import build_augmentations

# reuse the raw_volume base from cell 2b; reload as a tensor so augment() gets the expected shape.
m = meta
crop_h = crop_w = cfg.crop_size
base = ds._load_crop(int(m['t1']), int(m['z']), int(m['y']), int(m['x']), crop_h, crop_w)

augment = build_augmentations(cfg)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax in axes.flat:
    aug_volume = augment(base.clone())[0].numpy()  # (z, h, w)
    ax.imshow(mip(aug_volume), cmap='gray')
    ax.axis('off')
plt.suptitle('16 independent augmentations of the same crop (each panel is a z-MIP)')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 4: per-augmentation isolation. Each panel is a z-MIP of one augmentation mode applied alone.
import copy
import matplotlib.pyplot as plt


def isolate_aug(name):
    # disable every other augmentation, keep per-crop norm so visual ranges are comparable,
    # and force p=1.0 on the isolated one so we always see its effect.
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = (attr == name)
    c.aug_percrop_norm.enabled = True
    if name is not None:
        getattr(c, name).p = 1.0
    return build_augmentations(c)


def raw_aug():
    # only per-crop norm, no other augmentations, so the panel matches what the model would see at rest.
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = False
    return build_augmentations(c)


modes = [
    ('raw (norm only)', raw_aug()),
    ('intensity only',  isolate_aug('aug_intensity')),
    ('gamma only',      isolate_aug('aug_gamma')),
    ('noise only',      isolate_aug('aug_noise')),
    ('blur only',       isolate_aug('aug_blur')),
    ('full stack',      build_augmentations(cfg)),
]

fig, axes = plt.subplots(1, len(modes), figsize=(3 * len(modes), 3.5))
for ax, (title, aug) in zip(axes, modes):
    aug_volume = aug(base.clone())[0].numpy()  # (z, h, w)
    ax.imshow(mip(aug_volume), cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('per-augmentation isolation (each panel is a z-MIP)')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 5: crop-size sweep on one anchor; overlay rectangles on a MIP of the crop's z range.
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# mip across the crop's z range so the rectangles match the volume the model actually sees.
z0 = int(meta['z'])
z1 = z0 + cfg.z_thickness
frame = ds.arr[int(meta['t1']), cfg.channel, z0:z1, :, :].compute()
frame = np.asarray(frame, dtype=np.float32).max(axis=0)  # (y, x) mip across z
frame = np.clip(frame, 0, cfg.max_intensity) / cfg.max_intensity

x_off = int(meta['x']); y_off = int(meta['y'])
sizes = [64, 96, 128, 160, 224]
colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:blue', 'tab:purple']

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(frame, cmap='gray')
for s, c in zip(sizes, colors):
    rect = patches.Rectangle(
        (x_off, y_off), s, s,
        linewidth=2, edgecolor=c, facecolor='none', label=f'crop_size={s}',
    )
    ax.add_patch(rect)
ax.legend(loc='upper right')
ax.set_title(f'crop-size sweep at t={int(meta["t1"])}, MIP over z=[{z0}, {z1})')
ax.axis('off')
plt.tight_layout(); plt.show()
